# Neural Network from Scratch

Building a complete neural network using only NumPy.

**Topics:**
- Forward propagation
- Activation functions
- Backpropagation
- Training on XOR and spiral datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Activation Functions

Non-linear functions that allow neural networks to learn complex patterns.

In [ ]:
# Activation functions and their derivatives
class Activations:
    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    @staticmethod
    def sigmoid_deriv(z):
        s = Activations.sigmoid(z)
        return s * (1 - s)
    
    @staticmethod
    def relu(z):
        return np.maximum(0, z)
    
    @staticmethod
    def relu_deriv(z):
        return (z > 0).astype(float)
    
    @staticmethod
    def tanh(z):
        return np.tanh(z)
    
    @staticmethod
    def tanh_deriv(z):
        return 1 - np.tanh(z)**2

In [ ]:
# Visualize activations
z = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for ax, (name, fn) in zip(axes[0], [("Sigmoid", Activations.sigmoid), ("ReLU", Activations.relu), ("Tanh", Activations.tanh)]):
    ax.plot(z, fn(z), linewidth=2)
    ax.set_title(name); ax.axhline(0, color="k", linewidth=0.5); ax.axvline(0, color="k", linewidth=0.5)

for ax, (name, fn) in zip(axes[1], [("Sigmoid'", Activations.sigmoid_deriv), ("ReLU'", Activations.relu_deriv), ("Tanh'", Activations.tanh_deriv)]):
    ax.plot(z, fn(z), linewidth=2, color="orange")
    ax.set_title(name); ax.axhline(0, color="k", linewidth=0.5); ax.axvline(0, color="k", linewidth=0.5)

plt.suptitle("Activation Functions and Their Derivatives")
plt.tight_layout()
plt.show()

## 2. The Neural Network Class

Architecture: Input → Hidden layers (with ReLU) → Output (Sigmoid)

**Forward pass:** ^{[l]} = g(W^{[l]} a^{[l-1]} + b^{[l]})$

**Backward pass:** Compute gradients using chain rule

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_sizes, lr=0.01):
        """
        layer_sizes: list like [2, 8, 4, 1] for input=2, hidden=8,4, output=1
        """
        self.layer_sizes = layer_sizes
        self.lr = lr
        self.n_layers = len(layer_sizes) - 1
        self.losses = []
        
        # Initialize weights (He initialization)
        np.random.seed(42)
        self.weights = []
        self.biases = []
        for i in range(self.n_layers):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0/layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)
    
    def forward(self, X):
        """Forward propagation"""
        self.activations = [X]
        self.z_values = []
        
        a = X
        for i in range(self.n_layers):
            z = a @ self.weights[i] + self.biases[i]
            self.z_values.append(z)
            
            if i == self.n_layers - 1:  # Output layer
                a = Activations.sigmoid(z)
            else:  # Hidden layers
                a = Activations.relu(z)
            self.activations.append(a)
        
        return a
    
    def backward(self, y):
        """Backpropagation"""
        m = y.shape[0]
        y = y.reshape(-1, 1)
        
        # Output layer gradient
        delta = self.activations[-1] - y  # derivative of BCE + sigmoid
        
        for i in range(self.n_layers - 1, -1, -1):
            dw = (1/m) * self.activations[i].T @ delta
            db = (1/m) * np.sum(delta, axis=0, keepdims=True)
            
            if i > 0:
                delta = (delta @ self.weights[i].T) * Activations.relu_deriv(self.z_values[i-1])
            
            self.weights[i] -= self.lr * dw
            self.biases[i] -= self.lr * db
    
    def compute_loss(self, y_pred, y_true):
        y_true = y_true.reshape(-1, 1)
        eps = 1e-8
        return -np.mean(y_true*np.log(y_pred+eps) + (1-y_true)*np.log(1-y_pred+eps))
    
    def train(self, X, y, epochs=1000, verbose=True):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y_pred, y)
            self.losses.append(loss)
            self.backward(y)
            
            if verbose and (epoch+1) % (epochs//5) == 0:
                acc = ((y_pred.flatten() > 0.5) == y).mean()
                print(f"Epoch {epoch+1:4d} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")
    
    def predict(self, X):
        return (self.forward(X).flatten() > 0.5).astype(int)

## 3. Training on XOR Problem

XOR is not linearly separable - a single layer cannot solve it. We need at least one hidden layer.

In [ ]:
# XOR dataset
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

print("XOR Truth Table:")
for x, y in zip(X_xor, y_xor):
    print(f"  {x} -> {int(y)}")

In [ ]:
# Train on XOR
nn_xor = NeuralNetwork([2, 8, 4, 1], lr=0.5)
nn_xor.train(X_xor, y_xor, epochs=2000)

print(f"
Predictions:")
probs = nn_xor.forward(X_xor).flatten()
for x, y, p in zip(X_xor, y_xor, probs):
    print(f"  {x} -> pred: {p:.4f} (true: {int(y)})")

In [ ]:
# Visualize XOR decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
Z = nn_xor.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax1.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.7)
ax1.scatter(X_xor[:,0], X_xor[:,1], c=y_xor, cmap="RdBu", edgecolors="k", s=200, zorder=5)
ax1.set_title("XOR Decision Boundary")

ax2.plot(nn_xor.losses)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.set_title("Training Loss")
plt.tight_layout()
plt.show()

## 4. Spiral Dataset (Harder Problem)

A spiral dataset is much harder - it requires learning complex non-linear boundaries.

In [ ]:
# Generate spiral dataset
def make_spiral(n_points=200, noise=0.2):
    np.random.seed(42)
    n = n_points // 2
    
    # Class 0: spiral going one way
    theta0 = np.linspace(0, 3*np.pi, n)
    r0 = theta0 / (3*np.pi)
    x0 = r0 * np.cos(theta0) + np.random.randn(n)*noise*0.1
    y0 = r0 * np.sin(theta0) + np.random.randn(n)*noise*0.1
    
    # Class 1: spiral going other way
    theta1 = np.linspace(0, 3*np.pi, n)
    r1 = theta1 / (3*np.pi)
    x1 = -r1 * np.cos(theta1) + np.random.randn(n)*noise*0.1
    y1 = -r1 * np.sin(theta1) + np.random.randn(n)*noise*0.1
    
    X = np.vstack([np.c_[x0, y0], np.c_[x1, y1]])
    y = np.array([0]*n + [1]*n)
    return X, y

X_spiral, y_spiral = make_spiral(400)

plt.figure(figsize=(6, 6))
plt.scatter(X_spiral[:,0], X_spiral[:,1], c=y_spiral, cmap="RdBu", s=20, edgecolors="k", linewidths=0.5)
plt.title("Spiral Dataset")
plt.axis("equal")
plt.show()

In [ ]:
# Train on spiral
nn_spiral = NeuralNetwork([2, 32, 16, 1], lr=1.0)
nn_spiral.train(X_spiral, y_spiral, epochs=3000)

In [ ]:
# Visualize spiral decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
Z = nn_spiral.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax1.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.7)
ax1.scatter(X_spiral[:,0], X_spiral[:,1], c=y_spiral, cmap="RdBu", edgecolors="k", s=15, linewidths=0.5)
ax1.set_title("Spiral - Decision Boundary")
ax1.set_aspect("equal")

ax2.plot(nn_spiral.losses)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.set_title("Training Loss")
plt.tight_layout()
plt.show()

acc = (nn_spiral.predict(X_spiral) == y_spiral).mean()
print(f"Final accuracy: {acc:.4f}")

## 5. Visualizing What Neurons Learn

Let us look at the hidden layer activations to understand what each neuron represents.

In [ ]:
# Visualize first hidden layer neuron activations
nn_spiral.forward(np.c_[xx.ravel(), yy.ravel()])
hidden1 = nn_spiral.activations[1]  # First hidden layer

n_neurons = min(8, hidden1.shape[1])
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat[:n_neurons]):
    Z_neuron = hidden1[:, i].reshape(xx.shape)
    ax.contourf(xx, yy, Z_neuron, levels=20, cmap="viridis")
    ax.scatter(X_spiral[:,0], X_spiral[:,1], c=y_spiral, cmap="RdBu", s=5, edgecolors="k", linewidths=0.2)
    ax.set_title(f"Neuron {i}")
    ax.set_aspect("equal")

plt.suptitle("First Hidden Layer Neuron Activations")
plt.tight_layout()
plt.show()

## 6. Effect of Network Width and Depth

In [ ]:
# Compare architectures
architectures = [
    ([2, 4, 1], "Shallow (4)"),
    ([2, 16, 1], "Shallow (16)"),
    ([2, 8, 8, 1], "Deep (8-8)"),
    ([2, 32, 16, 1], "Deep (32-16)"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (arch, name) in zip(axes, architectures):
    nn = NeuralNetwork(arch, lr=1.0)
    nn.train(X_spiral, y_spiral, epochs=3000, verbose=False)
    Z = nn.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.7)
    ax.scatter(X_spiral[:,0], X_spiral[:,1], c=y_spiral, cmap="RdBu", s=10, edgecolors="k", linewidths=0.3)
    acc = (nn.predict(X_spiral) == y_spiral).mean()
    ax.set_title(f"{name}
acc={acc:.2%}")
    ax.set_aspect("equal")

plt.suptitle("Effect of Architecture on Learning")
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Forward prop** computes predictions layer by layer
2. **Backprop** uses chain rule to compute gradients efficiently
3. **Activation functions** add non-linearity (ReLU is most common)
4. **Width** helps learn more features per layer
5. **Depth** helps learn hierarchical features
6. **He initialization** prevents vanishing/exploding gradients

**Next:** Implement mini-batch SGD, dropout, and batch normalization!